In [1]:
!pip install ultralytics opencv-python-headless filterpy lap scipy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.1 MB/s eta 0:00:00


In [2]:
!wget -q -O sort.py https://raw.githubusercontent.com/abewley/sort/master/sort.py


In [3]:
# Fix matplotlib backend for Colab
with open("sort.py", "r") as f:
    content = f.read()
content = content.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
with open("sort.py", "w") as f:
    f.write(content)
print("SORT backend fixed.")


SORT backend fixed.


In [5]:
from google.colab import files
uploaded = files.upload()


Saving traffic.mp4 to traffic.mp4


## Enhanced Vehicle & Human Detection with SORT Tracking

In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
from sort import Sort
from google.colab.patches import cv2_imshow

# ============================================================
#  CONFIGURATION
# ============================================================
VIDEO_PATH   = "traffic.mp4"   # change to your uploaded filename
OUTPUT_PATH  = "output_enhanced.mp4"

CONF_THRESHOLD = 0.30   # lower = catch more objects (was 0.4)
NMS_IOU        = 0.45   # non-max suppression IOU

# --- Speed Control -------------------------------------------
# Process every Nth frame for detection but write EVERY frame.
# This effectively "slows down" output: detections linger longer.
# 1 = every frame (real-time speed)
# 2 = detect on every 2nd frame (smoother for fast videos)
DETECT_EVERY_N = 1       # set to 2 if your video is too fast

# Output FPS — reduce to slow down playback (original is 30)
OUTPUT_FPS = 15          # half speed output so details are visible

# ============================================================
#  CLASS CONFIGURATION
#  COCO class id -> (display_label, BGR_color)
# ============================================================
CLASS_CONFIG = {
    0:  ("Person",       (50,  50,  255)),  # red-ish
    2:  ("Car",          (50,  220, 50)),   # green
    3:  ("Motorcycle",   (0,   165, 255)),  # orange
    5:  ("Bus",          (255, 80,  80)),   # blue
    6:  ("Train",        (200, 0,   200)),  # purple
    7:  ("Truck",        (200, 200, 0)),    # cyan-yellow
}
TARGET_CLASSES = set(CLASS_CONFIG.keys())

# Auto-rickshaw heuristic: motorcycle-sized box with near-square aspect ratio
def refine_label(cls_id, x1, y1, x2, y2):
    """Return refined label for display."""
    w, h = x2 - x1, y2 - y1
    if cls_id == 3:  # Motorcycle
        aspect = w / max(h, 1)
        if 0.8 <= aspect <= 1.4:
            return "Auto/3-Wheeler"
    return CLASS_CONFIG[cls_id][0]

# ============================================================
#  SEPARATE TRACKERS per class group
#  (vehicles tracked together; humans separately)
# ============================================================
vehicle_tracker = Sort(max_age=25, min_hits=2, iou_threshold=0.30)
person_tracker  = Sort(max_age=20, min_hits=2, iou_threshold=0.30)

VEHICLE_CLASSES = {2, 3, 5, 6, 7}
PERSON_CLASS    = {0}

# ============================================================
#  LOAD MODEL  (yolov8s = small, better accuracy than nano)
# ============================================================
model = YOLO("yolov8s.pt")
print("Model loaded: YOLOv8s")

# ============================================================
#  VIDEO I/O
# ============================================================
cap = cv2.VideoCapture(VIDEO_PATH)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
orig_fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {frame_w}x{frame_h} @ {orig_fps:.1f}fps, {total_frames} frames")
print(f"Output FPS set to: {OUTPUT_FPS}  (playback speed: {OUTPUT_FPS/orig_fps:.2f}x)")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, OUTPUT_FPS, (frame_w, frame_h))

# ============================================================
#  TRACKING STATE
# ============================================================
tracked_vehicle_ids = set()
tracked_person_ids  = set()

# id -> label cache so label doesn't flicker between frames
id_label_cache = {}

# Last known tracked objects (reused on non-detect frames)
last_vehicle_tracks = np.empty((0, 5))
last_person_tracks  = np.empty((0, 5))

# id -> cls mapping (vehicle tracks don't carry cls, we infer from detection overlap)
# We store detections with cls for matching
last_det_boxes = []  # list of (x1,y1,x2,y2,cls_id,conf)

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / float(aA + aB - inter)

def match_track_to_class(track_box, det_boxes):
    """Find best matching detection class for a track box."""
    best_iou, best_cls = 0, 2  # default: car
    for (dx1, dy1, dx2, dy2, cls_id, _conf) in det_boxes:
        i = iou(track_box, (dx1, dy1, dx2, dy2))
        if i > best_iou:
            best_iou = i
            best_cls = cls_id
    return best_cls

# ============================================================
#  OVERLAY HELPERS
# ============================================================
def draw_box(frame, x1, y1, x2, y2, label, color, track_id):
    # Draw filled label background
    text = f"{label} #{int(track_id)}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    cv2.putText(frame, text, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

def draw_hud(frame, v_count, p_count, density, adaptive_green, frame_idx):
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (340, 220), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    def put(text, y, color=(255,255,255), scale=0.7, thick=2):
        cv2.putText(frame, text, (20, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick)

    put(f"Frame: {frame_idx}", 35)
    put(f"Vehicles: {v_count}",  65,  (100, 255, 100))
    put(f"Persons : {p_count}",  95,  (100, 180, 255))
    put(f"Density : {density}",  125, (255, 200, 50))
    put(f"Adaptive Green: {adaptive_green}s", 155, (50, 255, 255))

    total_unique = len(tracked_vehicle_ids) + len(tracked_person_ids)
    put(f"Total Unique IDs: {total_unique}", 185, (200, 200, 200), 0.6)

    if v_count + p_count > 10:
        cv2.putText(frame, "!! CONGESTION ALERT !!",
                    (frame_w//2 - 200, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)

# Density helper
def estimate_density(count):
    if count < 5: return "Low"
    if count < 10: return "Medium"
    return "High"

def adaptive_signal_time(count):
    return min(60, 10 + count * 2)

# ============================================================
#  MAIN LOOP
# ============================================================
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    run_detection = (frame_idx % DETECT_EVERY_N == 0)

    if run_detection:
        results = model(frame, conf=CONF_THRESHOLD, iou=NMS_IOU,
                        classes=list(TARGET_CLASSES), verbose=False)[0]

        v_dets, p_dets = [], []
        last_det_boxes = []

        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            last_det_boxes.append((x1, y1, x2, y2, cls_id, conf))

            entry = [x1, y1, x2, y2, conf]
            if cls_id in VEHICLE_CLASSES:
                v_dets.append(entry)
            elif cls_id in PERSON_CLASS:
                p_dets.append(entry)

        v_arr = np.array(v_dets) if v_dets else np.empty((0, 5))
        p_arr = np.array(p_dets) if p_dets else np.empty((0, 5))

        last_vehicle_tracks = vehicle_tracker.update(v_arr)
        last_person_tracks  = person_tracker.update(p_arr)

    # --- Draw vehicle tracks ---
    v_count = 0
    for track in last_vehicle_tracks:
        x1, y1, x2, y2, tid = map(int, track)
        tid = int(tid)
        tracked_vehicle_ids.add(tid)

        # Determine class label
        if tid not in id_label_cache:
            cls_id = match_track_to_class((x1, y1, x2, y2), last_det_boxes)
            label  = refine_label(cls_id, x1, y1, x2, y2)
            color  = CLASS_CONFIG.get(cls_id, ("Vehicle", (50,220,50)))[1]
            id_label_cache[tid] = (label, color)

        label, color = id_label_cache[tid]
        draw_box(frame, x1, y1, x2, y2, label, color, tid)
        v_count += 1

    # --- Draw person tracks ---
    p_count = 0
    for track in last_person_tracks:
        x1, y1, x2, y2, tid = map(int, track)
        tid = int(tid) + 10000  # offset to avoid ID clash with vehicles
        tracked_person_ids.add(tid)

        draw_box(frame, x1, y1, x2, y2, "Person",
                 CLASS_CONFIG[0][1], tid - 10000)
        p_count += 1

    # --- HUD overlay ---
    density      = estimate_density(v_count)
    adaptive_grn = adaptive_signal_time(v_count)
    draw_hud(frame, v_count, p_count, density, adaptive_grn, frame_idx)

    out.write(frame)

    if frame_idx % 150 == 0:
        print(f"  Processed {frame_idx}/{total_frames} frames "
              f"| V:{v_count} P:{p_count}")

cap.release()
out.release()

print("\n✅ Processing Complete!")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Unique vehicles tracked : {len(tracked_vehicle_ids)}")
print(f"   Unique persons  tracked : {len(tracked_person_ids)}")
print(f"   Total unique IDs        : {len(tracked_vehicle_ids)+len(tracked_person_ids)}")


Model loaded: YOLOv8s
Video: 1920x1080 @ 30.0fps, 1170 frames
Output FPS set to: 15  (playback speed: 0.50x)
  Processed 150/1170 frames | V:9 P:1
  Processed 300/1170 frames | V:5 P:4
  Processed 450/1170 frames | V:4 P:2
  Processed 600/1170 frames | V:6 P:7
  Processed 750/1170 frames | V:3 P:3
  Processed 900/1170 frames | V:4 P:1
  Processed 1050/1170 frames | V:14 P:5

✅ Processing Complete!
   Output: output_enhanced.mp4
   Unique vehicles tracked : 1023
   Unique persons  tracked : 508
   Total unique IDs        : 1531


In [9]:
from google.colab import files
files.download("output_enhanced.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>